# Superstore: Predicting Loss-Making Orders

**Goal:** predict whether an order will lose money (`Is Loss` = 1) from its details.

**Model:** Random Forest classifier.

**Before you run:** run the cells from top to bottom. The first cell asks you to upload `Superstore_step2.csv`.

## 1. Upload the data file

In [ ]:
# In Colab this opens a file picker. Choose Superstore_step2.csv
try:
    from google.colab import files
    uploaded = files.upload()
except ImportError:
    print("Not running in Colab: make sure Superstore_step2.csv is in the same folder.")

## 2. Load the data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("Superstore_step2.csv", encoding="utf-8")

print(df.shape)          # expected: (9993, 27)
df.head()

## 3. Quick look at the target

18.7% of orders are losses, so the classes are imbalanced.

In [ ]:
print(df["Is Loss"].value_counts())
print("Loss share:", round(df["Is Loss"].mean(), 3))

## 4. Discount vs losses

The clearest finding in the data.

In [ ]:
band_order = ["No Discount", "Low", "Medium", "High"]
loss_by_band = df.groupby("Discount Band")["Is Loss"].mean().reindex(band_order) * 100

loss_by_band.plot(kind="bar", color="#4C72B0")
plt.title("Loss rate by discount band")
plt.ylabel("Orders that lose money (%)")
plt.xlabel("")
plt.xticks(rotation=0)
plt.show()

print(loss_by_band.round(1))

## 5. Choose the input columns

Removed before training:
- **IDs and names** (`Order ID`, `Customer ID`, `Customer Name`, `Product ID`, `Product Name`, `Product Key`): they don't generalize to new orders.
- **Raw dates** (`Order Date`, `Ship Date`): we already have Month, Year, Week Day, Quarter, Ship Days.
- **`Profit`**: it *is* the answer (Is Loss comes from Profit), so keeping it would be cheating (leakage).
- **`Sales`**: we keep `Log Sales` instead.
- **`State`**: 49 values; we use `Region` for location.

In [ ]:
drop_cols = [
    "Order ID", "Customer ID", "Customer Name",
    "Product ID", "Product Name", "Product Key",
    "Order Date", "Ship Date",
    "Profit", "Sales", "State",
]

X = df.drop(columns=drop_cols + ["Is Loss"])
y = df["Is Loss"]

print(X.columns.tolist())

## 6. Encode the category columns

Models need numbers, so each category becomes its own 0/1 column (one-hot encoding).

In [ ]:
cat_cols = ["Ship Mode", "Segment", "Region", "Category", "Sub-Category", "Discount Band"]

X = pd.get_dummies(X, columns=cat_cols, dtype=int)

print(X.shape)
X.head()

## 7. Split into train and test

80% train, 20% test. `stratify=y` keeps the same loss share (18.7%) in both parts.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train:", X_train.shape, " loss share:", round(y_train.mean(), 3))
print("Test: ", X_test.shape,  " loss share:", round(y_test.mean(), 3))

## 8. Train the model

`class_weight="balanced"` gives more weight to the smaller class (losses), so the model doesn't ignore them.
Random Forest doesn't need scaling.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)
print("Model trained")

## 9. Evaluate

Look at **recall** and **F1** for class 1 (losses), not only accuracy.
- Recall: of all real losses, how many the model caught.
- Precision: of all orders the model flagged, how many were real losses.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=["Profit", "Loss"]))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 3))

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred), display_labels=["Profit", "Loss"]).plot(cmap="Blues")
plt.title("Confusion matrix (test data)")
plt.show()

## 10. What drives losses? (feature importance)

In [ ]:
importance = pd.Series(model.feature_importances_, index=X.columns).sort_values()

importance.tail(15).plot(kind="barh", color="#4C72B0", figsize=(8, 6))
plt.title("Top 15 features")
plt.xlabel("Importance")
plt.show()

## 11. Save the model

Downloads `loss_model.joblib` so you can use it later (for example in a dashboard).

In [ ]:
import joblib

joblib.dump(model, "loss_model.joblib")
joblib.dump(X.columns.tolist(), "model_columns.joblib")

try:
    from google.colab import files
    files.download("loss_model.joblib")
    files.download("model_columns.joblib")
except ImportError:
    print("Saved locally.")

## Results summary

- Discount is the main driver of losses: no order without a discount lost money, and every order with a discount above 40% did.
- Test results: recall 0.90 and F1 0.85 for losses, ROC-AUC 0.99. The model catches 9 out of 10 loss-making orders.
- Because discount explains losses so well, the scores are high. This is a real pattern in the data, not a bug.
- **Business recommendation:** limit discounts above 20%, especially on the sub-categories shown in the feature importance chart.